In [1]:
#!/usr/bin/env python
# coding: utf-8

import os
import pandas as pd
import numpy as np
from glob import glob

# ====================================
# CONFIGURATION
# ====================================
BASE_DIR = "CRSS"  # main folder
OUTPUT_PATH = "cleaned_data/crss_person_cleaned.csv"

VARS_OF_INTEREST_PERSON = [
    "PER_NO", "VEH_NO", "PER_TYP", "AGE", "SEX", "INJ_SEV",
    "DRINKING", "DRUGS", "REST_USE", "AIR_BAG", "SEAT_POS", "EJECTION",
    "CASENUM", "REGION", "URBANICITY", "STRATUM", "WEIGHT"
]

# ====================================
# CLEANING FUNCTIONS
# ====================================
def airbagclean(column):
    return column.where(~column.isin([0,97,98,99]), pd.NA)

def restuseclean(column):
    return column.where(~column.isin([0, 19, 20, 29, 96, 98, 99]), pd.NA)

def sexclean(column):
    return column.where(~column.isin([8,9]), pd.NA)

def injsevname(column):
    return column.where(~column.isin([5,6,9]), pd.NA)

def drinkclean(column):
    return column.where(~column.isin([8,9]), pd.NA)

def ageclean(column):
    column = column.where(~column.isin([998, 999]), pd.NA)
    column = column.where(column <= 120, pd.NA)
    return column

def drugclean(column):
    return column.where(~column.isin([8,9]), pd.NA)

def pertype(column):
    return column.where(~column.isin([9,19,13]), pd.NA)

def seatposname(column):
    return column.where(~column.isin([0, 98, 99]), pd.NA)

def ejection(column):
    return column.where(~column.isin([7,8,9]), pd.NA)

# ====================================
# LOAD & CLEAN PER YEAR
# ====================================
def load_crss_person_file(year_folder):
    """Load person.csv for a given year folder"""
    candidates = glob(os.path.join(year_folder, "person.csv"))
    if not candidates:
        print(f"No person file found in {year_folder}")
        return None
    file_path = candidates[0]
    try:
        df = pd.read_csv(file_path, usecols=VARS_OF_INTEREST_PERSON, encoding="utf-8", low_memory=False)
    except UnicodeDecodeError:
        df = pd.read_csv(file_path, usecols=VARS_OF_INTEREST_PERSON, encoding="latin1", low_memory=False)

    df.columns = df.columns.str.upper().str.strip()
    df["YEAR"] = int(os.path.basename(year_folder))
    return df

def clean_crss_person_df(df):
    df = df.copy()
    if "SEX" in df.columns: df["SEX"] = sexclean(df["SEX"])
    if "INJ_SEV" in df.columns: df["INJ_SEV"] = injsevname(df["INJ_SEV"])
    if "DRINKING" in df.columns: df["DRINKING"] = drinkclean(df["DRINKING"])
    if "AIR_BAG" in df.columns: df["AIR_BAG"] = airbagclean(df["AIR_BAG"])
    if "AGE" in df.columns: df["AGE"] = ageclean(df["AGE"])
    if "DRUGS" in df.columns: df["DRUGS"] = drugclean(df["DRUGS"])
    if "REST_USE" in df.columns: df["REST_USE"] = restuseclean(df["REST_USE"])
    if "PER_TYP" in df.columns: df["PER_TYP"] = pertype(df["PER_TYP"])
    if "SEAT_POS" in df.columns: df["SEAT_POS"] = seatposname(df["SEAT_POS"])
    if "EJECTION" in df.columns: df["EJECTION"] = ejection(df["EJECTION"])
    return df

# ====================================
# COMBINE ALL YEARS
# ====================================
def combine_crss_person_data(base_dir):
    year_folders = sorted([f.path for f in os.scandir(base_dir) if f.is_dir()])
    all_dfs = []

    for folder in year_folders:
        df = load_crss_person_file(folder)
        if df is None:
            continue

        df = clean_crss_person_df(df)

        # Create unique ID: YEAR + CASENUM
        if "CASENUM" in df.columns:
            df["ID"] = df["YEAR"].astype(str) + "_" + df["CASENUM"].astype(str)
            df.insert(0, "ID", df.pop("ID"))

        all_dfs.append(df)
        print(f"Processed {os.path.basename(folder)}: {df.shape[0]} rows, {df.shape[1]} cols")

    combined = pd.concat(all_dfs, ignore_index=True)
    return combined

# ====================================
# DATA QUALITY
# ====================================
def summarize_data_quality(df):
    print("\n=== DATA QUALITY SUMMARY ===")
    missing_pct = df.isna().mean() * 100
    print("Missingness (%):")
    print(missing_pct.sort_values(ascending=False))
    print("\nRanges / Unique Values:")
    for col in df.columns:
        if df[col].dtype in [np.int64, np.float64]:
            print(f"{col}: min={df[col].min()}, max={df[col].max()}")
        else:
            print(f"{col}: {df[col].nunique()} unique values")

# ====================================
# DUPLICATE CHECK
# ====================================
def check_duplicate_keys(df):
    key_cols = ["CASENUM", "VEH_NO", "PER_NO"]
    duplicates = df[df.duplicated(subset=key_cols, keep=False)]
    print(f"\nDuplicate key combinations ({', '.join(key_cols)}): {len(duplicates)}")
    return duplicates

# ====================================
# MAIN
# ====================================
if __name__ == "__main__":
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

    crss_person_clean = combine_crss_person_data(BASE_DIR)
    summarize_data_quality(crss_person_clean)

    # Duplicate key check
    check_duplicate_keys(crss_person_clean)

    # Save
    crss_person_clean.to_csv(OUTPUT_PATH, index=False)
    print(f"\nCleaned CRSS person data saved to: {OUTPUT_PATH}")


Processed 2016: 117759 rows, 19 cols
Processed 2017: 138913 rows, 19 cols
Processed 2018: 120230 rows, 19 cols
Processed 2019: 135410 rows, 19 cols
Processed 2020: 131962 rows, 19 cols
Processed 2021: 133734 rows, 19 cols
Processed 2022: 132175 rows, 19 cols
Processed 2023: 122388 rows, 19 cols

=== DATA QUALITY SUMMARY ===
Missingness (%):
DRUGS         50.016996
DRINKING      40.665969
REST_USE      17.732824
AIR_BAG       13.363246
EJECTION      11.475434
AGE            6.365954
SEAT_POS       5.046045
SEX            4.305273
INJ_SEV        3.738532
PER_TYP        0.026148
WEIGHT         0.000000
REGION         0.000000
URBANICITY     0.000000
ID             0.000000
CASENUM        0.000000
STRATUM        0.000000
PER_NO         0.000000
VEH_NO         0.000000
YEAR           0.000000
dtype: float64

Ranges / Unique Values:
ID: 417195 unique values
CASENUM: min=201600014311, max=202305779265
VEH_NO: min=0, max=15
PER_NO: min=1, max=75
STRATUM: min=2, max=10
AGE: min=0.0, max=120.0
S